In [1]:
# Deep Learning framework
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader, random_split
from torchsummary import summary

# Audio processing
import torchaudio
import torchaudio.transforms as T
import librosa

# Pre-trained image models
# import timm

# Play the audio in Jupyter notebook
from IPython.display import Audio
import pandas as pd
import os
import numpy as np

from scripts.dataset import AudioDataset
from scripts.get_dataframe import get_dataframe, get_balanced_dataframe
from models.cnn_tutorial import CNNNetworkTutorial

if torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(DEVICE)

cuda


In [2]:
class_mapping = ["positive", "negative"]

In [3]:
def train_single_epoch(model, dataloader, criterion, optimizer):
    for batch_idx, (inputs, labels) in enumerate(dataloader):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

        # Zero your gradients for every batch!
        optimizer.zero_grad()

        # Make predictions for this batch
        outputs = model.forward(inputs)
        #outputs = model(inputs)

        # Compute the loss and its gradients
        loss = criterion(outputs, labels)
        loss.backward()

        # Adjust learning weights
        optimizer.step()

    return loss


def train(model, dataloader, criterion, optimizer, epochs, train_set):
    for epoch in range(epochs):
        loss = train_single_epoch(model, dataloader, criterion, optimizer)
        #if epoch % (epochs // 1) == 0:
        print(f"iteration {epoch}: loss {loss.item()} | accuracy: {accuracy(model, train_set)}")
            # Get model weights after each iteration
            #model_weights = model.state_dict()
            #print("Model Weights:", model_weights)


def predict(model, input, target):
    model.eval()
    with torch.no_grad():
        predictions = model(input)
        # Tensor (1, 10) -> [ [0.1, 0.01, ..., 0.6] ]
        predicted_index = predictions[0].argmax(0)
        predicted = class_mapping[predicted_index]
        expected = class_mapping[target]
    return predicted, expected

def accuracy(model, dataset):
    correct = 0
    negative_predictions = 0
    total_samples = len(dataset)
    for idx in dataset.indices:
        input = dataset[dataset.indices.index(idx)][0].to(DEVICE)
        target = dataset[dataset.indices.index(idx)][1].argmax(0)
        input.unsqueeze_(0)

        predicted, expected = predict(model, input, target)
        #print(predicted, expected)
        if predicted == expected:
            correct += 1

        if predicted == "negative":
            negative_predictions += 1
    
    print((negative_predictions / total_samples) * 100, "negative predictions")
    return (correct / total_samples) * 100

In [4]:
BATCH_SIZE = 10
EPOCHS = 10
LEARNING_RATE = 0.0001
SAVE = False
TRAIN_RATIO = 0.9
l2_lambda = 0.01

df = get_balanced_dataframe()

dataset = AudioDataset(df)

# Calculate the lengths of training and testing subsets
train_length = int(len(dataset) * TRAIN_RATIO)

# Use random_split to split the dataset
train_set, validation_set = random_split(dataset, [train_length, len(dataset) - train_length])
print("train:", len(train_set))
print("validation:", len(validation_set))
# Create data loaders for training and testing
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
#validation_loader = DataLoader(validation_set, batch_size=BATCH_SIZE, shuffle=False)

# construct model and assign it to device
model = CNNNetworkTutorial().to(DEVICE)
#summary(model.cuda(), (1, 64, 44))

# initialise loss funtion + optimiser
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),
                                lr=LEARNING_RATE,
                                weight_decay=l2_lambda)

# train model
train(model, train_loader, criterion, optimizer, EPOCHS, train_set)

print("Train Set Accuracy: ", accuracy(model, train_set),"%")
print("Validation Set Accuracy: ", accuracy(model, validation_set),"%")

# save model
if SAVE:
    torch.save(model.state_dict(), "cnn_tutorial.pth")
    print("Saved as cnn_tutorial.pth")

train: 325
validation: 37
100.0 negative predictions
iteration 0: loss 0.9132618308067322 | accuracy: 49.23076923076923
100.0 negative predictions
iteration 1: loss 0.5132617354393005 | accuracy: 49.23076923076923
100.0 negative predictions
iteration 2: loss 0.5132617354393005 | accuracy: 49.23076923076923
100.0 negative predictions
iteration 3: loss 0.7132617831230164 | accuracy: 49.23076923076923
100.0 negative predictions
iteration 4: loss 0.7132617831230164 | accuracy: 49.23076923076923


KeyboardInterrupt: 

In [ ]:
input, target = dataset[10][0].to(DEVICE), dataset[10][1].argmax(0)
input.unsqueeze_(0)


predicted, expected = predict(model, input, target)
print(f"Predicted: '{predicted}', expected: '{expected}'")

Predicted: 'negative', expected: 'negative'
